# ConvNeXt-224 + ConvNeXt-320 + ResNet: three-voter experiment

Our current best submission is the original-only 224/320 probability blend, public
score **0.95039**. The teammate has supplied ONLY a binary submission, not ResNet
validation/test probabilities or weights. Consequently this notebook builds a
**hard-voting experiment**, not the probability ensemble in `cnn_2_ensemble_resnet.ipynb`.

Each label is voted on independently by three branches:

1. ConvNeXt-224: its original test probabilities >= its original OOF thresholds.
2. ConvNeXt-320-fast: its original test probabilities >= its original OOF thresholds.
3. ResNet: the teammate's supplied binary labels, unchanged.

Predict 1 when at least two branches vote 1. When ConvNeXt branches agree, ResNet
cannot change their decision; when they disagree, its vote decides. This does NOT
average 0/1 values as model probabilities. Moris and Motya may both remain positive;
no exclusivity or `keep_one` rule is imposed. Branch thresholds are reused as saved,
not fitted against the leaderboard or binary ResNet submission.

There are no ResNet validation predictions, so we cannot compute ensemble validation
F1, calibrate combined probabilities, or claim improvement. CSV disagreement counts
are diagnostic, not accuracy. Keep the 0.95039 submission and submit this as a separate
experiment. If ResNet weights or probabilities become available, use the probability
ensemble notebook instead.

Attach competition data, both ConvNeXt backup datasets, and the teammate's CSV.
No GPU, training, image loading, W&B or competition upload is performed here.

## Imports and paths
Known working folders are used first, then matching attached Input folders. Specify
paths explicitly if multiple compatible runs are attached. The teammate's CSV is
pinned by SHA so another submission is not silently selected.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

BASE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_ensemble')
FINE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_320_fast_ensemble')
BLEND_DIR = Path('/kaggle/working/convnext_blend_224_320_fast')
SAMPLE_PATH = Path('/kaggle/input/competitions/cats-and-dogs/sample_submission.csv')
RESNET_SUBMISSION_PATH = None  # set to the uploaded submission (6).csv, or discover it by SHA
OUR_BEST_SUBMISSION_PATH = None  # optional: exact downloaded file with public score 0.95039
OUTPUT_DIR = Path('/kaggle/working/convnext_resnet_vote')
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
EXPECTED_RESNET_SHA256 = '54845e37011bdce9ee0aa11dfe06f44bb68d1dc9208d8a12a9a9f113fa4a05e1'


## Validate sources and align IDs
The prediction manifests must refer to the checkpoints used to create the selected
blend. All rows are aligned by image_id. No labels from sample_submission are used
as ground truth. The optional known best submission is a reference for reproducing
our own baseline, not for threshold tuning.

In [ ]:
def read_unique(path):
    frame = pd.read_csv(path, dtype={'image_id': str})
    assert len(frame) and frame.image_id.notna().all() and frame.image_id.is_unique, str(path)
    return frame

def align(frame, ids):
    ids = list(ids)
    assert len(ids) == len(set(ids)) and set(frame.image_id) == set(ids), 'Image IDs differ.'
    return frame.set_index('image_id').loc[ids].reset_index()

def probabilities(frame):
    values = frame[LABEL_COLS].to_numpy(dtype=np.float64)
    assert values.shape == (len(frame), len(LABEL_COLS)) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all(), 'Expected probabilities in [0, 1].'
    return values

def cutoffs(values):
    values = np.asarray(values, dtype=np.float64)
    assert values.shape == (len(LABEL_COLS),) and np.isfinite(values).all()
    assert ((values > 0) & (values < 1)).all()
    return values

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def input_directories():
    for root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        if root.is_dir():
            for directory, subdirs, filenames in os.walk(root):
                subdirs[:] = [name for name in subdirs if name not in {'train', 'test', 'images', 'wandb', '.virtual_documents'}]
                yield Path(directory), set(filenames)

def resolve_directory(preferred, required, predicate):
    preferred = Path(preferred)
    if all((preferred / name).is_file() for name in required):
        assert predicate(preferred), f'Incompatible source: {preferred}'
        return preferred
    candidates = [directory for directory, files in input_directories()
                  if set(required).issubset(files) and predicate(directory)]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the backup or set the directory explicitly. Required: {required}. Candidates: {candidates}')
    return candidates[0]

def matches_resolution(directory, image_size):
    cfg = json.loads((directory / 'ensemble_config.json').read_text())
    return (cfg.get('model_name') == 'convnext_tiny.fb_in1k'
            and cfg.get('preprocessing', {}).get('image_size') == image_size)

BASE_PREDICTIONS_DIR = resolve_directory(BASE_PREDICTIONS_DIR,
    ['ensemble_config.json', 'test_probabilities_original.csv'], lambda p: matches_resolution(p, 224))
FINE_PREDICTIONS_DIR = resolve_directory(FINE_PREDICTIONS_DIR,
    ['ensemble_config.json', 'test_probabilities_original.csv'], lambda p: matches_resolution(p, 320))
base_config = json.loads((BASE_PREDICTIONS_DIR / 'ensemble_config.json').read_text())
fine_config = json.loads((FINE_PREDICTIONS_DIR / 'ensemble_config.json').read_text())
for cfg in [base_config, fine_config]:
    assert cfg['label_cols'] == LABEL_COLS and cfg['folds'] == list(range(5))
    assert cfg['aggregation'] == 'equal_mean_sigmoid_probabilities'
    assert set(cfg['checkpoint_hashes']) == {str(fold) for fold in range(5)}
assert base_config['folds_fingerprint'] == fine_config['folds_fingerprint']

def matches_blend(directory):
    cfg = json.loads((directory / 'blend_config.json').read_text())
    return (cfg.get('base_cv_fingerprint') == base_config['cv_fingerprint']
            and cfg.get('fine_cv_fingerprint') == fine_config['cv_fingerprint'])

BLEND_DIR = resolve_directory(BLEND_DIR, ['blend_config.json'], matches_blend)
blend_config = json.loads((BLEND_DIR / 'blend_config.json').read_text())
assert blend_config['label_cols'] == LABEL_COLS and blend_config['weights'] == {'224': 0.5, '320': 0.5}
if not SAMPLE_PATH.is_file():
    candidates = [directory / 'sample_submission.csv' for directory, files in input_directories()
                  if {'train.csv', 'test.csv', 'sample_submission.csv'}.issubset(files)]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set SAMPLE_PATH to the competition sample_submission.csv: {candidates}')
    SAMPLE_PATH = candidates[0]
sample = read_unique(SAMPLE_PATH)
assert set(sample.columns) == {'image_id', *LABEL_COLS}
if RESNET_SUBMISSION_PATH is None:
    candidates = [directory / name for directory, files in input_directories() for name in sorted(files)
                  if name.lower().endswith('.csv') and name.lower().startswith('submission')
                  and file_digest(directory / name) == EXPECTED_RESNET_SHA256]
    if not candidates:
        raise FileNotFoundError('Upload the teammate submission (6).csv and set RESNET_SUBMISSION_PATH if needed.')
    RESNET_SUBMISSION_PATH = candidates[0]  # identical copies are safe because SHA matches
RESNET_SUBMISSION_PATH = Path(RESNET_SUBMISSION_PATH)
assert file_digest(RESNET_SUBMISSION_PATH) == EXPECTED_RESNET_SHA256, 'A different ResNet submission was supplied.'
resnet = align(read_unique(RESNET_SUBMISSION_PATH), sample.image_id)
assert set(resnet.columns) == set(sample.columns) and resnet[LABEL_COLS].isin([0, 1]).all().all()
base_path = BASE_PREDICTIONS_DIR / 'test_probabilities_original.csv'
fine_path = FINE_PREDICTIONS_DIR / 'test_probabilities_original.csv'
base = align(read_unique(base_path), sample.image_id)
fine = align(read_unique(fine_path), sample.image_id)
p224, p320 = probabilities(base), probabilities(fine)
t224 = cutoffs(base_config['thresholds']['original'])
t320 = cutoffs(fine_config['thresholds']['original'])
t_blend = cutoffs(blend_config['thresholds']['original'])
our_baseline = ((p224 + p320) / 2 >= t_blend).astype(np.int32)
reference_path = Path(OUR_BEST_SUBMISSION_PATH) if OUR_BEST_SUBMISSION_PATH is not None else BLEND_DIR / 'submission_blend_224_320.csv'
if reference_path.is_file():
    reference = align(read_unique(reference_path), sample.image_id)
    assert reference[LABEL_COLS].isin([0, 1]).all().all()
    assert np.array_equal(reference[LABEL_COLS].to_numpy(), our_baseline), 'The selected sources do not reproduce our reference blend.'
    print('Our reference submission reproduced exactly.')
print('Aligned test rows:', len(sample), '| ResNet file:', RESNET_SUBMISSION_PATH.name)


## Vote and export
Use one fixed rule: two or more positive votes out of three. No thresholds or
weights are searched using this submission. Summary columns count differences
from our current best blend and from the colleague, not errors or improvements.

In [ ]:
vote224 = (p224 >= t224).astype(np.int32)
vote320 = (p320 >= t320).astype(np.int32)
vote_resnet = resnet[LABEL_COLS].to_numpy(dtype=np.int32)
vote_count = vote224 + vote320 + vote_resnet
ensemble = (vote_count >= 2).astype(np.int32)
assert np.array_equal(ensemble[vote224 == vote320], vote224[vote224 == vote320])
assert np.array_equal(ensemble[vote224 != vote320], vote_resnet[vote224 != vote320])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export(filename, predicted):
    submission = sample.copy()
    submission[LABEL_COLS] = predicted
    assert len(submission) == len(sample) and not submission.isna().any().any()
    assert submission[LABEL_COLS].isin([0, 1]).all().all()
    path = OUTPUT_DIR / filename
    submission.to_csv(path, index=False)
    print('Saved:', path)
    display(FileLink(str(path)))
    return path

baseline_path = export('submission_convnext_baseline.csv', our_baseline)
ensemble_path = export('submission_vote_convnext_resnet.csv', ensemble)
rows, changes = [], []
for j, label in enumerate(LABEL_COLS):
    changed = ensemble[:, j] != our_baseline[:, j]
    rows.append({'label': label, 'convnext224_positives': int(vote224[:, j].sum()),
        'convnext320_positives': int(vote320[:, j].sum()), 'resnet_positives': int(vote_resnet[:, j].sum()),
        'our_blend_positives': int(our_baseline[:, j].sum()), 'ensemble_positives': int(ensemble[:, j].sum()),
        'convnext_branches_disagree': int((vote224[:, j] != vote320[:, j]).sum()),
        'changed_from_our_blend': int(changed.sum()),
        'changed_0_to_1': int(((our_baseline[:, j] == 0) & (ensemble[:, j] == 1)).sum()),
        'changed_1_to_0': int(((our_baseline[:, j] == 1) & (ensemble[:, j] == 0)).sum()),
        'different_from_resnet': int((ensemble[:, j] != vote_resnet[:, j]).sum())})
    for index in np.flatnonzero(changed):
        changes.append({'image_id': sample.image_id.iloc[index], 'label': label,
            'prob224': float(p224[index, j]), 'prob320': float(p320[index, j]),
            'vote224': int(vote224[index, j]), 'vote320': int(vote320[index, j]),
            'resnet_vote': int(vote_resnet[index, j]), 'our_blend': int(our_baseline[index, j]),
            'ensemble_vote': int(ensemble[index, j])})
summary = pd.DataFrame(rows)
changes_frame = pd.DataFrame(changes, columns=['image_id', 'label', 'prob224', 'prob320', 'vote224',
    'vote320', 'resnet_vote', 'our_blend', 'ensemble_vote'])
summary.to_csv(OUTPUT_DIR / 'vote_summary.csv', index=False)
changes_frame.to_csv(OUTPUT_DIR / 'changed_predictions.csv', index=False)
display(summary)
print('Changed images relative to our blend:', int((ensemble != our_baseline).any(axis=1).sum()))
print('Changed label decisions:', int((ensemble != our_baseline).sum()))
metadata = {
    'schema': 'convnext_resnet_hard_vote_v1', 'label_cols': LABEL_COLS,
    'rule': 'at_least_two_positive_votes_out_of_three_per_label',
    'voters': ['convnext224_oof_thresholds', 'convnext320_oof_thresholds', 'resnet_binary_submission'],
    'tta': False, 'resnet_validation_available': False, 'independent_validation_score': None,
    'public_reference_score_reported_by_user': 0.95039,
    'base_cv_fingerprint': base_config['cv_fingerprint'], 'fine_cv_fingerprint': fine_config['cv_fingerprint'],
    'base_checkpoint_hashes': base_config['checkpoint_hashes'], 'fine_checkpoint_hashes': fine_config['checkpoint_hashes'],
    'folds_fingerprint': base_config['folds_fingerprint'],
    'thresholds': {'224': t224.tolist(), '320': t320.tolist(), 'our_blend': t_blend.tolist()},
    'input_hashes': {'base_test': file_digest(base_path), 'fine_test': file_digest(fine_path),
                     'blend_config': file_digest(BLEND_DIR / 'blend_config.json'),
                     'resnet_submission': file_digest(RESNET_SUBMISSION_PATH)},
    'output_hashes': {'baseline': file_digest(baseline_path), 'ensemble': file_digest(ensemble_path)},
    'changed_images': int((ensemble != our_baseline).any(axis=1).sum()),
    'changed_label_decisions': int((ensemble != our_baseline).sum()),
}
(OUTPUT_DIR / 'vote_config.json').write_text(json.dumps(metadata, indent=2))
print('Experimental file for manual Kaggle submission: submission_vote_convnext_resnet.csv')
print('No validation F1 is available for this voting ensemble; keep the previous best submission.')
